# Tratamento das features socioeconômicas — IPVS e Censo IBGE 2022

Analisa a cobertura, trata ausências e salva uma cópia da base H3 com as features socioeconômicas preparadas. A base original não é sobrescrita.

## Estratégia adotada

A cobertura é avaliada em um registro por setor censitário (`CD_SETOR`), pois os dados socioeconômicos se repetem nas linhas da série temporal. Zeros são mantidos como valores válidos. Para cada feature numérica é criado um indicador `<feature>_missing` antes da imputação. Valores ausentes são preenchidos pela mediana dos setores observados no mesmo município (`CIDADE`); se não houver valor municipal, usa-se a mediana global por setor. `N_IPVS` recebe a categoria `Não informado` e um indicador próprio. Nenhuma variável é descartada automaticamente: as features com maior ausência são mantidas com sinalização, para posterior avaliação no treino. A imputação não recupera o valor real, apenas viabiliza o uso da base.

In [1]:
import pandas as pd
from pathlib import Path
input_path = Path('../datasets/database_processed_h3.csv')
output_path = Path('../datasets/database_processed_h3_features_socio.csv')


In [2]:
socio_cols = ['C_IPVS', 'pop_total', 'domicilios_total', 'media_moradores', 'domicilios_vagos', 'pop_masc', 'pop_fem', 'jovens_15_19', 'jovens_20_24', 'jovens_25_29', 'alfabetizados_15mais', 'nao_alfabetizados_15mais', 'pop_branca', 'pop_preta', 'pop_parda', 'dom_casa', 'dom_apartamento', 'dom_cortico', 'dom_degradado', 'dom_agua_encanada', 'dom_esgoto_rede', 'dom_lixo_coletado', 'pct_jovens_15_29', 'taxa_alfabetizacao', 'pct_negros']


## Diagnóstico de ausências e zeros

O diagnóstico usa setores únicos para evitar que setores com mais linhas sejam sobreponderados.

In [3]:
sector_cols = ['CD_SETOR', 'CIDADE', 'N_IPVS'] + socio_cols
sector_df = pd.read_csv(input_path, usecols=sector_cols).drop_duplicates('CD_SETOR').copy()
missing_report = pd.DataFrame({
    'n_ausentes': sector_df[socio_cols].isna().sum(),
    'pct_ausentes': (sector_df[socio_cols].isna().mean() * 100).round(2),
    'n_zeros': sector_df[socio_cols].eq(0).sum(),
    'pct_zeros': (sector_df[socio_cols].eq(0).mean() * 100).round(2),
}).sort_values('pct_ausentes', ascending=False)
display(missing_report)
display(sector_df['N_IPVS'].isna().value_counts(normalize=True).rename('proporcao'))


,n_ausentes,pct_ausentes,n_zeros,pct_zeros
nao_alfabetizados_15mais,7798,17.98,4081,9.41
taxa_alfabetizacao,7798,17.98,0,0.00
dom_apartamento,3841,8.86,20445,47.15
dom_cortico,3597,8.30,37260,85.93
dom_casa,2716,6.26,4138,9.54
C_IPVS,2420,5.58,0,0.00
pct_negros,2283,5.26,13,0.03
pop_preta,2241,5.17,371,0.86
pct_jovens_15_29,2119,4.89,13,0.03
jovens_15_19,1701,3.92,186,0.43


N_IPVS
False    1.0
Name: proporcao, dtype: float64

## Imputar e salvar

A leitura e a escrita em blocos preservam todas as linhas e colunas sem exigir que a base inteira caiba em memória.

In [4]:
city_medians = {}
for col in socio_cols:
    city_medians[col] = sector_df.groupby('CIDADE')[col].median()
    global_median = sector_df[col].median()
    if pd.isna(global_median):
        raise ValueError(f'Não há valores observados para {col}.')
    city_medians[col] = city_medians[col].fillna(global_median)

first_chunk = True
for chunk in pd.read_csv(input_path, chunksize=200_000):
    for col in socio_cols:
        chunk[f'{col}_missing'] = chunk[col].isna().astype('int8')
        chunk[col] = chunk[col].fillna(chunk['CIDADE'].map(city_medians[col]))
    chunk['N_IPVS_missing'] = chunk['N_IPVS'].isna().astype('int8')
    chunk['N_IPVS'] = chunk['N_IPVS'].fillna('Não informado')
    chunk.to_csv(output_path, index=False, mode='w' if first_chunk else 'a', header=first_chunk)
    first_chunk = False
print(f'Base salva em {output_path.resolve()}')


Base salva em C:\Users\Usuario\Documents\USP\8_SEMESTRE\Resolução de Problemas II\sp-crime-spatiotemporal-prediction\datasets\database_processed_h3_features_socio.csv


## Validação

Verifica que as features numéricas não tenham nulos restantes e informa quantos valores foram imputados.

In [5]:
remaining_nulls = pd.Series(0, index=socio_cols, dtype='int64')
imputed = pd.Series(0, index=socio_cols, dtype='int64')
rows = 0
usecols = socio_cols + [f'{c}_missing' for c in socio_cols]
for chunk in pd.read_csv(output_path, usecols=usecols, chunksize=200_000):
    rows += len(chunk)
    remaining_nulls += chunk[socio_cols].isna().sum()
    imputed += chunk[[f'{c}_missing' for c in socio_cols]].sum().set_axis(socio_cols)
print('Linhas:', rows)
print('Nulos restantes:', int(remaining_nulls.sum()))
display((imputed / rows * 100).round(2).rename('% imputado'))


Linhas: 1739458
Nulos restantes: 0


C_IPVS                       7.08
pop_total                    0.00
domicilios_total             0.00
media_moradores              0.00
domicilios_vagos             0.00
pop_masc                     3.60
pop_fem                      3.60
jovens_15_19                 5.82
jovens_20_24                 4.73
jovens_25_29                 4.48
alfabetizados_15mais         3.60
nao_alfabetizados_15mais    21.80
pop_branca                   3.61
pop_preta                    6.49
pop_parda                    3.82
dom_casa                     7.63
dom_apartamento             10.38
dom_cortico                 11.04
dom_degradado                4.88
dom_agua_encanada            3.61
dom_esgoto_rede              3.94
dom_lixo_coletado            3.73
pct_jovens_15_29             6.65
taxa_alfabetizacao          21.80
pct_negros                   6.64
Name: % imputado, dtype: float64